# 6 — Integration prep: manifest, contract tables, vocabulary

Brings both modalities to the shared cell-table contract and works out what pairs with what.

Everything here is registration-free, so it runs before any images are mounted and produces
a real, checkable result on its own: the pairing table and the lineage/marker crosswalk.

Needs the integration environment (`environment-integration.yml`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from phenocycler import load_config

cfg = load_config(pathlib.Path.cwd().parent / 'config.ini')
print('mode           :', cfg.integration_mode)
print('fixed frame    :', cfg.fixed_modality)
print('data dir       :', cfg.data_dir)
print('panel explorer :', cfg.panel_explorer, '(exists:', cfg.panel_explorer.exists(), ')')

## S0 — pairing manifest

`donor_id` is the join key. Two things it has to fix: no Xenium artifact carries the key at
all (they are keyed by slide serial), and one slide serial can hold both a pancreas and a
lymph-node region — so the section key is `{serial}__{roi}`.

Rows reported as `donor_unknown` are Xenium runs whose donor is recorded nowhere upstream.
Fill them in at `data/integration/donor_overrides.csv` to bring them into the analysis.

In [ ]:
from phenocycler.integration.manifest import build_manifest, write_manifest, paired_rows

man, summary = build_manifest(cfg)
print(summary.render())
write_manifest(cfg, man)

In [ ]:
# Unscoped: every paired section in the cohort, both tissues.
paired = paired_rows(man)
paired[['donor_id', 'roi', 'tissue', 'disease_status', 'xenium_sample_key', 'xenium_serial']]


## S1 — both modalities onto the contract

`export_pheno` recovers two things the 8-class call collapses, without re-running anything:

* `immune_subclass` from the surviving CD3e/CD20/CD163 gates — Xenium resolves T/B/myeloid
  separately, so PhenoCycler needs to as well;
* `endocrine_subtype` from the hormone `_norm` argmax — the Endocrine class lumps
  INS/GCG/SST/CD99 together, which masks β-loss behind α-persistence.

In [ ]:
from phenocycler.integration.export_pheno import run_export_pheno

run_export_pheno(cfg, roi='panc')

In [ ]:
from phenocycler.integration.import_xenium import run_import_xenium

run_import_xenium(cfg, roi='panc')

## S2 — vocabulary crosswalk

Both pipelines call eight broad classes, but they are not the same partition of the tissue.
The crosswalk is derived from the pinned XeniumPanelExplorer submodule rather than hardcoded.

Watch the off-panel gates: INS, GCG, SST and Vimentin have no gene on the Xenium panel, so
those lineage comparisons go through surrogate identity panels — orthogonal evidence rather
than a 1:1 check.

In [ ]:
from phenocycler.integration.vocab import (load_panel_taxonomy, protein_gene_pairs,
                                            coverage_report, run_vocab)
from phenocycler.integration.contract import discover_partitions, read_cell_table, feature_name

panel = None
parts = discover_partitions(cfg.cells_xen_dir)
if parts:
    d, r = parts[0]
    t = read_cell_table(cfg.cells_xen_dir, d, r, modality='xenium')
    panel = [feature_name(c) for c in t.features if not feature_name(c).startswith('score_')]
    print(f'validating against {len(panel)} genes actually present in {d}/{r}')

crosswalk = run_vocab(cfg, panel)

In [ ]:
crosswalk[crosswalk.kind == 'lineage'][
    ['common', 'phenocycler', 'xenium', 'pheno_resolution', 'xenium_resolution', 'collapsed']]

### Sanity check

`export_pheno` should have recovered the immune split and the endocrine subtypes. If
`immune_subclass` is all empty, the RESTORE gates did not merge — check that
`restore_gated_redsea/` and `..._extra/` both exist for the donor.

In [ ]:
from phenocycler.integration.contract import discover_partitions, read_cell_table

for donor, roi in discover_partitions(cfg.cells_pheno_dir)[:1]:
    t = read_cell_table(cfg.cells_pheno_dir, donor, roi, modality='phenocycler')
    print(f'donor {donor}: {t.n_cells:,} cells, {len(t.features)} marker features')
    print()
    print(t.df.groupby(['lineage_native', 'lineage_common']).size())
    print()
    print('immune_subclass  :', t.df.loc[t.df.lineage_native == 'Immune',
                                         'immune_subclass'].value_counts().to_dict())
    print('endocrine_subtype:', t.df.loc[t.df.lineage_native == 'Endocrine',
                                         'endocrine_subtype'].value_counts().to_dict())
    print('evidence_positive:', t.df.evidence_positive.value_counts().to_dict())